In [1]:
import os
import numpy as np
import random
import warnings
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Set seeds for reproducibility
os.environ['PYTHONHASHSEED'] = str(1)
np.random.seed(1)
random.seed(1)
warnings.filterwarnings("ignore")

# 1. Data Loading
data = pd.read_excel('dataset.xlsx')  # Ensure the file path is correct
X = data.iloc[:, :-1].copy()
y = data.iloc[:, -1].copy()

# Ensure data is numeric
X = X.apply(pd.to_numeric, errors='coerce')
y = pd.to_numeric(y, errors='coerce')

# Check for invalid values
if X.isnull().any().any() or y.isnull().any():
    raise ValueError("Dataset contains invalid values (NaN or inf). Please check your data.")

# 2. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1
)

# 3. Hyperparameter Selection (Elastic Net with CV)
# 使用标准化 + ElasticNetCV；通过CV选择 alpha 与 l1_ratio
enet_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("enet", ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.99, 1.0],
        alphas=np.logspace(-4, 2, 50),
        cv=5,
        max_iter=20000,
        n_jobs=-1,
        random_state=None  # ElasticNetCV 对默认坐标下降为确定性（selection='cyclic'）
    ))
])

enet_pipe.fit(X_train, y_train)

best_alpha = float(enet_pipe.named_steps["enet"].alpha_)
best_l1_ratio = float(np.atleast_1d(enet_pipe.named_steps["enet"].l1_ratio_)[0])
print(f"Best parameters (Elastic Net): alpha={best_alpha:.6g}, l1_ratio={best_l1_ratio:.3f}")

# 4. Model Evaluation
def evaluate_model(model, X_train, X_test, y_train, y_test):
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    metrics = {
        'Train RMSE': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'Train R^2': r2_score(y_train, y_pred_train),
        'Test R^2': r2_score(y_test, y_pred_test),
        'Train MAE': mean_absolute_error(y_train, y_pred_train),
        'Test MAE': mean_absolute_error(y_test, y_pred_test)
    }
    return metrics

results = evaluate_model(enet_pipe, X_train, X_test, y_train, y_test)
for metric, value in results.items():
    print(f'{metric}: {value}')

# 5. Export Results
def export_results(y_train, y_pred_train, y_test, y_pred_test, filename='Results_ElasticNet.xlsx'):
    with pd.ExcelWriter(filename) as writer:
        train_results_df = pd.DataFrame({
            'y_true': y_train.values,
            'y_pred': y_pred_train
        })
        test_results_df = pd.DataFrame({
            'y_true': y_test.values,
            'y_pred': y_pred_test
        })
        train_results_df.to_excel(writer, sheet_name='Train Results', index=False)
        test_results_df.to_excel(writer, sheet_name='Test Results', index=False)

export_results(
    y_train,
    enet_pipe.predict(X_train),
    y_test,
    enet_pipe.predict(X_test)
)

# （可选）导出系数方便论文分析
coef = enet_pipe.named_steps["enet"].coef_
intercept = enet_pipe.named_steps["enet"].intercept_
coef_df = pd.DataFrame({"feature": X.columns, "coef": coef})
coef_df.to_excel("ElasticNet_Coefficients.xlsx", index=False)
print("Intercept:", intercept)


Best parameters (Elastic Net): alpha=1.9307, l1_ratio=0.990
Train RMSE: 19.72686482551938
Test RMSE: 19.67724233721787
Train R^2: 0.7474238642817523
Test R^2: 0.7738337940987434
Train MAE: 15.989639683630504
Test MAE: 16.933663973142085
Intercept: 79.96872340425533
